In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


# ============================================================================
# DATASET: EIT / EMG / CAP -> HEIGHT CLASS  (NO segment_id)
#   - window-level classification (one label per window)
# ============================================================================

class CrossModalHeightClsDatasetNoSeg(Dataset):
    """
    Loads EIT/EMG/CAP windows and predicts HEIGHT as a classification label.

    Expected columns:
      - time_sec (optional)
      - EIT: eit_ch1..eit_ch8
      - EMG: emg0..emg3
      - CAP: cap0_ma
      - HEIGHT target: auto-detect from candidates

    Target strategy (window-level):
      - Take the most frequent label (mode) within the window.
        (Good when height is constant per task segment.)
    """

    def __init__(
        self,
        csv_path,
        seq_length=50,
        stride=10,
        drop_nan_target=True,
        fill_input_nan_value=0.0,
        height_col_candidates=(
            "height_class", "touch_height", "height", "touchHeight", "height_level", "h"
        ),
        max_unique_for_direct_classes=50,
        binning_if_continuous="quantile",  # "quantile" or "uniform"
        num_bins=5,
    ):
        self.seq_length = int(seq_length)
        self.stride = int(stride)

        df = pd.read_csv(csv_path)

        # --- detect height column ---
        height_col = None
        for c in height_col_candidates:
            if c in df.columns:
                height_col = c
                break
        if height_col is None:
            # fuzzy fallback
            for c in df.columns:
                if "height" in c.lower():
                    height_col = c
                    break

        if height_col is None:
            raise ValueError(
                f"No HEIGHT column found. Tried: {height_col_candidates} and fuzzy 'height*'.\n"
                f"Available columns: {list(df.columns)}"
            )
        self.height_col = height_col

        # --- sensor columns ---
        self.eit_cols = [f"eit_ch{i}" for i in range(1, 9)]
        self.emg_cols = [f"emg{i}" for i in range(4)]
        self.cap_cols = ["cap0_ma"]

        for c in self.eit_cols + self.emg_cols + self.cap_cols + [self.height_col]:
            if c not in df.columns:
                raise ValueError(f"Missing expected column '{c}' in {csv_path}")

        # sort by time if available
        if "time_sec" in df.columns:
            df = df.sort_values("time_sec").reset_index(drop=True)
        else:
            df = df.reset_index(drop=True)

        # numeric conversion
        all_cols = self.eit_cols + self.emg_cols + self.cap_cols + [self.height_col]
        for c in all_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        # optional: drop rows where target is NaN
        if drop_nan_target:
            df = df[df[self.height_col].notna()].copy()

        # arrays
        eit = df[self.eit_cols].to_numpy(dtype=np.float32)
        emg = df[self.emg_cols].to_numpy(dtype=np.float32)
        cap = df[self.cap_cols].to_numpy(dtype=np.float32)
        y_raw = df[self.height_col].to_numpy(dtype=np.float32)  # keep float for mapping

        # replace inf with nan
        for arr in (eit, emg, cap):
            arr[np.isinf(arr)] = np.nan
        y_raw[np.isinf(y_raw)] = np.nan

        # drop rows where ALL inputs are NaN
        valid_mask = (
            np.isfinite(eit).any(axis=1) |
            np.isfinite(emg).any(axis=1) |
            np.isfinite(cap).any(axis=1)
        )
        eit, emg, cap, y_raw = eit[valid_mask], emg[valid_mask], cap[valid_mask], y_raw[valid_mask]

        # fill NaNs in inputs (important: avoid NaNs in forward/loss)
        eit = np.nan_to_num(eit, nan=fill_input_nan_value, posinf=fill_input_nan_value, neginf=fill_input_nan_value)
        emg = np.nan_to_num(emg, nan=fill_input_nan_value, posinf=fill_input_nan_value, neginf=fill_input_nan_value)
        cap = np.nan_to_num(cap, nan=fill_input_nan_value, posinf=fill_input_nan_value, neginf=fill_input_nan_value)

        # ---- Build class mapping ----
        y_finite = y_raw[np.isfinite(y_raw)]
        if y_finite.size == 0:
            raise ValueError("All height targets are NaN/inf after cleaning.")

        uniq = np.unique(y_finite)

        # If already discrete-ish (few unique values), use direct classes
        if len(uniq) <= max_unique_for_direct_classes:
            # Map unique values -> class ids
            self.class_values = uniq.tolist()
            value_to_id = {v: i for i, v in enumerate(self.class_values)}

            # convert y_raw -> y_id (NaNs kept as -1)
            y_id = np.full_like(y_raw, fill_value=-1, dtype=np.int64)
            for v, i in value_to_id.items():
                y_id[np.isfinite(y_raw) & (y_raw == v)] = i

        else:
            # Otherwise treat it as continuous and bin into classes
            if binning_if_continuous == "quantile":
                edges = np.quantile(y_finite, np.linspace(0, 1, num_bins + 1))
            else:
                edges = np.linspace(np.min(y_finite), np.max(y_finite), num_bins + 1)

            # avoid duplicate edges
            edges = np.unique(edges)
            if len(edges) < 3:
                raise ValueError("Binning failed: height distribution too degenerate.")

            self.class_values = [f"bin_{i}" for i in range(len(edges) - 1)]
            y_id = np.full_like(y_raw, fill_value=-1, dtype=np.int64)
            finite_mask = np.isfinite(y_raw)
            y_id[finite_mask] = np.clip(np.digitize(y_raw[finite_mask], edges[1:-1], right=False), 0, len(edges) - 2)

        self.num_classes = len(self.class_values)

        # ---- Windowing (window-level label = mode) ----
        self.sequences = []
        n = len(y_id)
        if n < self.seq_length:
            return

        num_windows = (n - self.seq_length) // self.stride + 1
        for i in range(num_windows):
            s = i * self.stride
            e = s + self.seq_length

            eit_win = eit[s:e]
            emg_win = emg[s:e]
            cap_win = cap[s:e]
            y_win = y_id[s:e]

            # skip windows with any invalid target
            y_win_valid = y_win[y_win >= 0]
            if y_win_valid.size == 0:
                continue

            # mode label for the window
            label = int(np.bincount(y_win_valid).argmax())

            self.sequences.append({
                "eit": eit_win.astype(np.float32),
                "emg": emg_win.astype(np.float32),
                "cap": cap_win.astype(np.float32),
                "label": label
            })

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return {
            "eit": torch.tensor(seq["eit"], dtype=torch.float32),   # [T,8]
            "emg": torch.tensor(seq["emg"], dtype=torch.float32),   # [T,4]
            "cap": torch.tensor(seq["cap"], dtype=torch.float32),   # [T,1]
            "y":   torch.tensor(seq["label"], dtype=torch.long),    # scalar class id
        }


# ============================================================================
# ENCODERS + FUSION
# ============================================================================

class EITEncoder(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return h[-1]  # [B,H]


class EMGEncoder(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return h[-1]


class CapEncoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return h[-1]


class FusionFlexible(nn.Module):
    def __init__(self, encoder_hidden=64, shared_size=128, dropout=0.3):
        super().__init__()
        self.encoder_hidden = encoder_hidden
        self.fc = nn.Linear(encoder_hidden * 3, shared_size)
        self.bn = nn.BatchNorm1d(shared_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, feats):
        B = feats[0].shape[0]
        device = feats[0].device
        fused = torch.zeros(B, 3 * self.encoder_hidden, device=device)
        for i, f in enumerate(feats[:3]):
            fused[:, i*self.encoder_hidden:(i+1)*self.encoder_hidden] = f
        x = self.drop(torch.relu(self.bn(self.fc(fused))))
        return x


# ============================================================================
# CLASSIFIER MODEL
# ============================================================================

class FlexibleMultiModalToHeightClsModel(nn.Module):
    """
    Window-level classification: outputs logits [B, C]
    """
    def __init__(self,
                 num_classes,
                 input_modalities=('eit', 'emg', 'cap'),
                 encoder_hidden=64,
                 encoder_layers=2,
                 shared_size=128,
                 dropout=0.3):
        super().__init__()
        self.input_modalities = list(input_modalities)

        self.eit_encoder = EITEncoder(8, encoder_hidden, encoder_layers, dropout)
        self.emg_encoder = EMGEncoder(4, encoder_hidden, encoder_layers, dropout)
        self.cap_encoder = CapEncoder(1, encoder_hidden, encoder_layers, dropout)

        self.fusion = FusionFlexible(encoder_hidden, shared_size, dropout)
        self.classifier = nn.Linear(shared_size, num_classes)

    def forward(self, eit=None, emg=None, cap=None):
        feats = []
        if 'eit' in self.input_modalities:
            feats.append(self.eit_encoder(eit))
        if 'emg' in self.input_modalities:
            feats.append(self.emg_encoder(emg))
        if 'cap' in self.input_modalities:
            feats.append(self.cap_encoder(cap))

        shared = self.fusion(feats)
        logits = self.classifier(shared)  # [B,C]
        return logits


# ============================================================================
# TRAIN / EVAL
# ============================================================================

def train_epoch_cls(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    losses = []

    for batch in tqdm(loader, desc="Training (→ HEIGHT CLS)"):
        eit = batch["eit"].to(device)
        emg = batch["emg"].to(device)
        cap = batch["cap"].to(device)
        y   = batch["y"].to(device)

        logits = model(eit=eit, emg=emg, cap=cap)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses)) if losses else 0.0


@torch.no_grad()
def eval_cls(model, loader, device):
    model.eval()
    all_pred, all_true = [], []
    losses = []
    criterion = nn.CrossEntropyLoss()

    for batch in tqdm(loader, desc="Testing (→ HEIGHT CLS)"):
        eit = batch["eit"].to(device)
        emg = batch["emg"].to(device)
        cap = batch["cap"].to(device)
        y   = batch["y"].to(device)

        logits = model(eit=eit, emg=emg, cap=cap)
        loss = criterion(logits, y)
        losses.append(loss.item())

        pred = torch.argmax(logits, dim=1)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    cm  = confusion_matrix(y_true, y_pred)

    return {
        "loss": float(np.mean(losses)) if losses else 0.0,
        "accuracy": float(acc),
        "macro_f1": float(f1m),
        "confusion_matrix": cm
    }


# ============================================================================
# RUN: 70% train / 30% test
# ============================================================================

INPUT_MODALITIES = ['cap', 'eit']  # same style as your regression runs
seq_length = 30
stride = 2
batch_size = 32
num_epochs = 200
learning_rate = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PATH = "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_4_height_processed.csv"  # <-- change if needed

dataset = CrossModalHeightClsDatasetNoSeg(
    DATA_PATH,
    seq_length=seq_length,
    stride=stride,
    drop_nan_target=True,
    fill_input_nan_value=0.0,
)

print("Detected height column:", dataset.height_col)
print("Num classes:", dataset.num_classes)
print("Class values (mapping id->value):", dataset.class_values[:20], "..." if dataset.num_classes > 20 else "")
print("Total windows:", len(dataset))

if len(dataset) == 0:
    raise ValueError("Dataset has 0 windows. Try reducing seq_length or increasing stride.")

n_total = len(dataset)
n_train = int(0.7 * n_total)
n_test  = n_total - n_train

train_set, test_set = random_split(
    dataset, [n_train, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)

model = FlexibleMultiModalToHeightClsModel(
    num_classes=dataset.num_classes,
    input_modalities=INPUT_MODALITIES,
    encoder_hidden=64,
    encoder_layers=2,
    shared_size=128,
    dropout=0.3
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

best_test_acc = -1.0
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    tr_loss = train_epoch_cls(model, train_loader, optimizer, device)
    metrics = eval_cls(model, test_loader, device)
    print(f"  Train loss: {tr_loss:.6f}")
    print(f"  Test  loss: {metrics['loss']:.6f} | acc: {metrics['accuracy']:.4f} | macro-F1: {metrics['macro_f1']:.4f}")

    if metrics["accuracy"] > best_test_acc:
        best_test_acc = metrics["accuracy"]
        torch.save(
            {
                "model_state": model.state_dict(),
                "num_classes": dataset.num_classes,
                "class_values": dataset.class_values,
                "height_col": dataset.height_col,
                "input_modalities": INPUT_MODALITIES,
                "seq_length": seq_length,
            },
            "best_height_cls_model.pth"
        )
        print("  ✓ Best model saved!")

print("\nBest test acc:", best_test_acc)
